In [11]:
import ROOT
ROOT.EnableImplicitMT(16)
import pandas as pd
import libPy

weights = [
    "hw_nominal",     # nominal MC weight.                  scalar
    "hw_alphaS_up",   # up alpha_s variaton for PHD4LHC;    scalar
    "hw_alphaS_dn",   # down alpha_s variaton for PHD4LHC;  scalar
    "hw_pdf4lhc_unc", # 30 Eigen variation for PHD4LHC      vector
    "hw_qcd",         # muR/muF variation for the given MC; vector
]

In [12]:
dfs = {}
dfs['ALL']      = ROOT.RDataFrame("tree", "ntuples/mc23_ggf_hyy_stxs.root")
dfs['UNKNOWN']  = dfs['ALL'].Filter("HTXS_Stage1_2_Fine_Category_pTjet30 == 0")

for val, category in libPy.stage_1_2_fine['ggf'].items():
    dfs[category] = dfs['ALL'].Filter(f"HTXS_Stage1_2_Fine_Category_pTjet30 == {val}")

# # determine the length of the vector weights, which should be the same for all events
# len_hw_pdf4lhc_unc = set(dfs['ALL'].Range(10).Define('len_hw_pdf4lhc_unc', 'hw_pdf4lhc_unc.size()').AsNumpy(['len_hw_pdf4lhc_unc'])['len_hw_pdf4lhc_unc'])
# len_hw_qcd         = set(dfs['ALL'].Range(10).Define('len_hw_qcd', 'hw_qcd.size()').AsNumpy(['len_hw_qcd'])['len_hw_qcd'])
# assert (len(len_hw_pdf4lhc_unc) == 1 and len(len_hw_qcd) == 1)
# len_hw_pdf4lhc_unc = list(len_hw_pdf4lhc_unc)[0]
# len_hw_qcd         = list(len_hw_qcd)[0]
# print(f"Length of hw_pdf4lhc_unc: {len_hw_pdf4lhc_unc}, Length of hw_qcd: {len_hw_qcd}")
len_hw_pdf4lhc_unc, len_hw_qcd = 41, 8

In [13]:
weight_dict = {}
futures = []
for slice, df in dfs.items():
    weight_dict[slice] = {}
    for weight in weights:
        if weight == "hw_pdf4lhc_unc":
            for i in range(len_hw_pdf4lhc_unc):
                weight_name = f"{weight}_{i}"
                weight_dict[slice][weight_name] = df.Define(weight_name, f"{weight}.at({i})").Filter(f"{weight_name} == {weight_name}").Sum(weight_name)
                futures.append(weight_dict[slice][weight_name])
        elif weight == "hw_qcd":
            for i in range(len_hw_qcd):
                weight_name = f"{weight}_{i}"
                weight_dict[slice][weight_name] = df.Define(weight_name, f"{weight}.at({i})").Filter(f"{weight_name} == {weight_name}").Sum(weight_name)
                futures.append(weight_dict[slice][weight_name])
        else:
            weight_dict[slice][weight] = df.Filter(f"{weight} == {weight}").Sum(weight)
            futures.append(weight_dict[slice][weight])
ROOT.RDF.RunGraphs(futures)

1

In [14]:
for slice, weight_sum_dict in weight_dict.items():
    for weight_name, weight_sum in weight_sum_dict.items():
        weight_dict[slice][weight_name] = weight_sum.GetValue()

In [15]:
pdf = pd.DataFrame(weight_dict)
pdf = pdf.apply(lambda row : (row / row['ALL']), axis=1)
ratio_pdf = pdf.apply(lambda row : row / pdf.iloc[0], axis=1)
# pdf.columns = [col + '_acc' for col in pdf.columns]
# pdf[[col.split('_')[0] + '_xs' for col in pdf.columns]] = pdf.apply(lambda row : row * xs, axis=1)
# pdf = pd.concat([pdf, ratio_pdf.add_suffix('_ratio')], axis=1)
pdf.to_csv("res_stxs/run3_ggf_stxs.csv", index=True)